
# Assignment 2: Restaurant Inspection Analysis
Course:CST 3512
Student:Ranphy Reyes
.


In [1]:


import pandas as pd
import numpy as np

# === EDIT THESE TWO VARIABLES ===
DATA_PATH = "DOHMH_New_York_City_Restaurant_Inspection_Results.csv"  # <-- put the CSV path provided by the class
RESTAURANT_KEYWORDS = ["Shake Shack"]  # e.g., ["Shake Shack"] or ["Chipotle", "Mexican Grill"]

# Read CSV (expecting standard NYC DOHMH schema)
df = pd.read_csv(DATA_PATH, dtype=str).replace({"": np.nan})
# Coerce numeric/date types where appropriate
for col in ["SCORE"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Preview
print("Rows:", len(df))
df.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'DOHMH_New_York_City_Restaurant_Inspection_Results.csv'

## 1) Restaurant Selection

In [ ]:

# Select a single restaurant using DBA name keywords (case-insensitive substring match).
# We also keep CAMIS (unique id) to avoid mixing same DBA across different locations.
mask = pd.Series([True] * len(df))
if "DBA" in df.columns:
    mask = mask & df["DBA"].str.contains("|".join([pd.regex.escape(k) for k in RESTAURANT_KEYWORDS]), case=False, na=False)

restaurant_df = df.loc[mask].copy()

# If multiple CAMIS are present (chain locations), keep the most frequent CAMIS to focus analysis.
if "CAMIS" in restaurant_df.columns and restaurant_df["CAMIS"].nunique() > 1:
    top_camis = (restaurant_df["CAMIS"].value_counts().idxmax())
    restaurant_df = restaurant_df[restaurant_df["CAMIS"] == top_camis].copy()

chosen_name = restaurant_df["DBA"].dropna().mode().iat[0] if "DBA" in restaurant_df.columns and not restaurant_df.empty else "N/A"
chosen_zip = restaurant_df["ZIPCODE"].dropna().mode().iat[0] if "ZIPCODE" in restaurant_df.columns and not restaurant_df.empty else "N/A"
chosen_cuisine = restaurant_df["CUISINE DESCRIPTION"].dropna().mode().iat[0] if "CUISINE DESCRIPTION" in restaurant_df.columns and not restaurant_df.empty else "N/A"

print(f"Chosen Restaurant: {chosen_name}")
print(f"ZIP Code: {chosen_zip}")
print(f"Cuisine: {chosen_cuisine}")

restaurant_df.head(5)


## 2) Data Extraction: all inspection & violation rows for the chosen restaurant

In [ ]:

# Keep only inspection/violation-relevant columns when available
cols_pref = [
    "CAMIS","DBA","BORO","BUILDING","STREET","ZIPCODE","PHONE","CUISINE DESCRIPTION",
    "INSPECTION DATE","ACTION","VIOLATION CODE","VIOLATION DESCRIPTION","CRITICAL FLAG",
    "SCORE","GRADE","GRADE DATE","RECORD DATE"
]
cols = [c for c in cols_pref if c in restaurant_df.columns]
subset = restaurant_df[cols].sort_values(by=["INSPECTION DATE","SCORE"], ascending=[True, True])
print("Filtered rows:", len(subset))
subset.head(10)


## 3) Research Questions

### Q1) How many **inspections** and **violations** has this restaurant received across the dataset? *(Condition / Query requirement)*

In [ ]:

# We'll count distinct inspection dates (proxy for count of inspections) and number of non-null violation codes
q1_df = subset.copy()

inspections = q1_df["INSPECTION DATE"].dropna().nunique() if "INSPECTION DATE" in q1_df.columns else np.nan
violations = q1_df["VIOLATION CODE"].notna().sum() if "VIOLATION CODE" in q1_df.columns else np.nan

print({"total_inspections": int(inspections) if pd.notna(inspections) else None,
       "total_violations": int(violations) if pd.notna(violations) else None})



**Interpretation:** The counts approximate how often the health department inspected this location and how many violations were recorded in total. A higher inspection count with relatively fewer violations suggests consistent compliance; the opposite may warrant attention.


### Q2) What are the **most frequent violation descriptions** for this restaurant? *(GroupBy / Pivot requirement)*

In [ ]:

q2_df = subset.copy()
if "VIOLATION DESCRIPTION" in q2_df.columns:
    top_violations = (
        q2_df["VIOLATION DESCRIPTION"]
        .value_counts(dropna=True)
        .head(10)
        .rename_axis("VIOLATION DESCRIPTION")
        .reset_index(name="COUNT")
    )
    top_violations
else:
    print("Column VIOLATION DESCRIPTION not found.")



**Interpretation:** These are the patterns most commonly cited by inspectors. Repeated temperature, hygiene, or pest-related issues point to process controls the restaurant should prioritize.


### Q3) How does this restaurant’s **average inspection score** compare with **same-cuisine restaurants in the same ZIP code**? *(Comparison requirement)*

In [ ]:

# Comparison: same ZIP + same cuisine
if all(col in df.columns for col in ["ZIPCODE","CUISINE DESCRIPTION","SCORE"]):
    comp_mask = (df["ZIPCODE"] == chosen_zip) & (df["CUISINE DESCRIPTION"] == chosen_cuisine)
    peer_df = df.loc[comp_mask].copy()
    peer_df["SCORE"] = pd.to_numeric(peer_df["SCORE"], errors="coerce")

    this_avg = pd.to_numeric(subset["SCORE"], errors="coerce").mean()
    peer_avg = peer_df["SCORE"].mean()

    print({"restaurant_avg_score": float(this_avg) if pd.notna(this_avg) else None,
           "peer_avg_score_zip_samecuisine": float(peer_avg) if pd.notna(peer_avg) else None,
           "peer_sample_size": int(peer_df["CAMIS"].nunique()) if "CAMIS" in peer_df.columns else int(len(peer_df))})
else:
    print("Missing one of ZIPCODE / CUISINE DESCRIPTION / SCORE columns.")



**Interpretation:** Lower average scores generally indicate fewer or less severe violations (better). If this restaurant’s average is lower than peers in the same ZIP and cuisine, it is outperforming local competitors on inspections.


### (Optional) Trend: average score by year (pivot)

In [ ]:

q4_df = subset.copy()
if "INSPECTION DATE" in q4_df.columns and "SCORE" in q4_df.columns:
    q4_df["year"] = q4_df["INSPECTION DATE"].dt.year
    trend = (q4_df
             .pivot_table(index="year", values="SCORE", aggfunc="mean")
             .reset_index()
             .sort_values("year"))
    trend
else:
    print("Need INSPECTION DATE and SCORE to compute yearly trend.")



**Interpretation:** A downward trend over years suggests improving compliance; upward may indicate deteriorating food-safety practices or inconsistent controls.


## 4) Export filtered rows (optional for your appendix)

In [ ]:

subset_out = "filtered_rows_for_submission.csv"
subset.to_csv(subset_out, index=False)
print(f"Saved: {subset_out}")



## ✅ Submission Checklist
- [ ] Notebook named like `your_last_name_HW2.ipynb` (e.g., `Galan_HW2.ipynb`).
- [ ] Restaurant name & location visible (see Section 1 cell output).
- [ ] Section 2 shows only the chosen restaurant’s rows.
- [ ] Three original questions:
  - Q1 uses a **condition/query**.
  - Q2 uses **groupby/pivot**.
  - Q3 includes a **comparison** vs. peer restaurants.
- [ ] Each question includes: code + output + a brief interpretation.
- [ ] Optional: attach `filtered_rows_for_submission.csv` as an appendix artifact.
